# Cyber Breach Prediction — 16-Feature Model Training Pipeline

This notebook trains a Random Forest classifier on **16 features** (reduced from the original 40) to predict whether an in-progress cyber intrusion will result in a successful breach (`Attack_Success = 1`) or be contained (`Attack_Success = 0`).

**16 Features:**
- Categorical (2): `Attack_Stage`, `Company_Size`
- Numeric/Binary (14): `Lateral_Movement`, `Privilege_Escalation`, `Persistence`, `Credential_Stolen`, `Data_Exfiltration_GB`, `Data_Encrypted`, `Phishing_Click`, `Detection_Time_Min`, `Response_Time_Min`, `Firewall`, `MFA`, `EDR`, `CVSS_Score`, `Patch_Age_Days`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report,
                             roc_curve, ConfusionMatrixDisplay)
import joblib
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load Raw Dataset

In [ ]:
df = pd.read_csv('../data/Enterprise_Cyber_Kill_Chain_Dataset.csv')
print(f'Raw shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 3. Data Cleaning

### 3.1 Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Dropped {before - len(df)} duplicate rows -> {len(df)} rows remain')

### 3.2 Remove Post-Breach Leakage Columns

These columns are only known **after** a breach is confirmed and would leak the target variable if included.

In [ ]:
LEAKAGE_COLS = [
    'Recovery_Cost_USD', 'Records_Compromised', 'Downtime_Hours',
    'Financial_Loss_USD', 'Cyber_Risk_Score', 'Risk_Level', 'Incident_Severity',
]
DROP_COLS = ['Incident_ID', 'Timestamp']

df = df.drop(columns=[c for c in LEAKAGE_COLS + DROP_COLS if c in df.columns])
print(f'Shape after dropping leakage/ID columns: {df.shape}')

### 3.3 Impute Missing Values

`Detection_Time_Min` has missing values; impute with the column median.

In [ ]:
print('Missing values before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

median_detect = df['Detection_Time_Min'].median()
df['Detection_Time_Min'] = df['Detection_Time_Min'].fillna(median_detect)
print(f'\nImputed Detection_Time_Min with median = {median_detect:.1f}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

## 4. Feature Selection (16 Features)

Feature-importance analysis on the full 40-feature model showed that ~24 features contribute almost no predictive signal. We retain the **16 most impactful features**.

In [ ]:
KEEP_FEATURES = [
    'Attack_Stage', 'Company_Size',           # Categorical
    'Lateral_Movement', 'Privilege_Escalation', 'Persistence',
    'Credential_Stolen', 'Data_Exfiltration_GB', 'Data_Encrypted',
    'Phishing_Click', 'Detection_Time_Min', 'Response_Time_Min',
    'Firewall', 'MFA', 'EDR', 'CVSS_Score', 'Patch_Age_Days',
]
TARGET = 'Attack_Success'

CATEGORICAL_FEATURES = ['Attack_Stage', 'Company_Size']
NUMERICAL_FEATURES = [f for f in KEEP_FEATURES if f not in CATEGORICAL_FEATURES]

X = df[KEEP_FEATURES].copy()
y = df[TARGET].copy()

print(f'Feature matrix: {X.shape}')
print(f'Target distribution:\n{y.value_counts()}')
print(f'\nBreach rate: {y.mean():.1%}')

## 5. Exploratory Data Analysis

### 5.1 Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
y.value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_xticklabels(['Contained (0)', 'Breach (1)'], rotation=0)
ax.set_title('Target Variable Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### 5.2 Breach Rate by Kill Chain Stage

In [ ]:
stage_order = ['Reconnaissance', 'Initial Access', 'Execution', 'Persistence', 'Impact']
breach_by_stage = df.groupby('Attack_Stage')[TARGET].mean().reindex(stage_order) * 100

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8b0000']
breach_by_stage.plot(kind='bar', color=colors, ax=ax)
ax.set_title('Breach Rate by Kill Chain Stage')
ax.set_ylabel('Breach Rate (%)')
ax.set_xlabel('Attack Stage')
ax.set_xticklabels(stage_order, rotation=30, ha='right')
for i, v in enumerate(breach_by_stage):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Defense Controls vs. Breach Rate

In [ ]:
defense_cols = ['Firewall', 'MFA', 'EDR']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, defense_cols):
    rates = df.groupby(col)[TARGET].mean() * 100
    rates.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
    ax.set_title(f'{col} vs. Breach Rate')
    ax.set_ylabel('Breach Rate (%)')
    ax.set_xticklabels(['Off (0)', 'On (1)'], rotation=0)
    for i, v in enumerate(rates):
        ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 Correlation Heatmap (Numeric Features)

In [ ]:
corr_data = df[NUMERICAL_FEATURES + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix (Numeric Features + Target)')
plt.tight_layout()
plt.show()

## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]} rows')
print(f'Test set:  {X_test.shape[0]} rows')
print(f'Train breach rate: {y_train.mean():.1%}')
print(f'Test breach rate:  {y_test.mean():.1%}')

## 7. Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ],
    remainder='passthrough',
)
print('Categorical features:', CATEGORICAL_FEATURES)
print('Numerical features:', NUMERICAL_FEATURES)
print(f'Total input features: {len(KEEP_FEATURES)}')

## 8. Model Training

### 8.1 Logistic Regression (Baseline)

In [ ]:
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_proba)

print(f'Logistic Regression Baseline:')
print(f'  Accuracy:     {lr_acc:.4f}')
print(f'  F1 (breach):  {lr_f1:.4f}')
print(f'  ROC-AUC:      {lr_auc:.4f}')

### 8.2 Random Forest (Primary Model)

In [ ]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

rf_pipe.fit(X_train, y_train)
rf_pred = rf_pipe.predict(X_test)
rf_proba = rf_pipe.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print(f'Random Forest:')
print(f'  Accuracy:     {rf_acc:.4f}')
print(f'  F1 (breach):  {rf_f1:.4f}')
print(f'  ROC-AUC:      {rf_auc:.4f}')

## 9. Model Evaluation

### 9.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, ax=axes[0],
                                         cmap='Blues', display_labels=['Contained', 'Breach'])
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, ax=axes[1],
                                         cmap='Oranges', display_labels=['Contained', 'Breach'])
axes[1].set_title('Random Forest')

plt.tight_layout()
plt.show()

### 9.2 Classification Report

In [ ]:
print('Random Forest Classification Report:')
print(classification_report(y_test, rf_pred, target_names=['Contained', 'Breach']))

### 9.3 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

ax.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC={lr_auc:.3f})', linewidth=2)
ax.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC={rf_auc:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Model Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 9.4 Model Comparison Summary

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [lr_acc, rf_acc],
    'F1 (Breach)': [lr_f1, rf_f1],
    'ROC-AUC': [lr_auc, rf_auc],
})
comparison.style.format({'Accuracy': '{:.4f}', 'F1 (Breach)': '{:.4f}', 'ROC-AUC': '{:.4f}'})

## 10. Feature Importance

In [ ]:
rf_model = rf_pipe.named_steps['classifier']
ohe = rf_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_feature_names = cat_feature_names + NUMERICAL_FEATURES
importances = rf_model.feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.plot(kind='barh', ax=ax, color='#00B4D8')
ax.set_title('Feature Importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nFeature importances (descending):')
for name, imp in feat_imp.sort_values(ascending=False).items():
    print(f'  {name:35s} {imp:.4f}')

## 11. Export Trained Model

In [ ]:
MODEL_PATH = '../model/cyber_model.joblib'
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(rf_pipe, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')
print(f'File size: {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB')

## 12. Smoke Test

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

# Case A: Contained attack (Reconnaissance, hardened defenses)
case_a = pd.DataFrame([{
    'Attack_Stage': 'Reconnaissance', 'Company_Size': 'Large',
    'Lateral_Movement': 0, 'Privilege_Escalation': 0, 'Persistence': 0,
    'Credential_Stolen': 0, 'Data_Exfiltration_GB': 0.0, 'Data_Encrypted': 0,
    'Phishing_Click': 0, 'Detection_Time_Min': 25, 'Response_Time_Min': 10,
    'Firewall': 1, 'MFA': 1, 'EDR': 1, 'CVSS_Score': 3.0, 'Patch_Age_Days': 10,
}])
prob_a = loaded_model.predict_proba(case_a)[0, 1]
print(f'Case A (contained) -> Breach probability: {prob_a:.4f} ({prob_a*100:.1f}%)')

# Case B: Critical breach (Impact, weak defenses)
case_b = pd.DataFrame([{
    'Attack_Stage': 'Impact', 'Company_Size': 'Small',
    'Lateral_Movement': 1, 'Privilege_Escalation': 1, 'Persistence': 1,
    'Credential_Stolen': 1, 'Data_Exfiltration_GB': 85.0, 'Data_Encrypted': 1,
    'Phishing_Click': 1, 'Detection_Time_Min': 210, 'Response_Time_Min': 150,
    'Firewall': 0, 'MFA': 0, 'EDR': 0, 'CVSS_Score': 9.4, 'Patch_Age_Days': 160,
}])
prob_b = loaded_model.predict_proba(case_b)[0, 1]
print(f'Case B (critical)  -> Breach probability: {prob_b:.4f} ({prob_b*100:.1f}%)')

## Summary

| Metric | Logistic Regression (Baseline) | Random Forest (Primary) |
|---|---|---|
| Accuracy | ~86.0% | ~86.2% |
| F1 (breach) | ~0.76 | ~0.77 |
| ROC-AUC | ~0.93 | ~0.94 |

The 16-feature model performs within 0.3% of the original 40-feature model while being significantly simpler and more interpretable. The kill chain stage remains the single strongest predictor of breach success.